In [1]:
!pip install -q unsloth trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/

In [2]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kaustubh-aggarwal31 (kaustubh-aggarwal31-vellore-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
!cp -r "/content/drive/MyDrive/llama3.1-8b-legal-clause-lora-sft" .
!ls llama3.1-8b-legal-clause-lora-sft

adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json


In [5]:
!cp "/content/drive/MyDrive/dpo_pairs.jsonl" .

In [6]:
!ls -la llama3.1-8b-legal-clause-lora-sft/
!ls -la dpo_pairs.jsonl

total 180792
drwx------ 2 root root      4096 Jul 11 17:13 .
drwxr-xr-x 1 root root      4096 Jul 11 17:13 ..
-rw------- 1 root root      1256 Jul 11 17:13 adapter_config.json
-rw------- 1 root root 167832240 Jul 11 17:13 adapter_model.safetensors
-rw------- 1 root root      4568 Jul 11 17:13 chat_template.jinja
-rw------- 1 root root      5262 Jul 11 17:13 README.md
-rw------- 1 root root     50668 Jul 11 17:13 tokenizer_config.json
-rw------- 1 root root  17209920 Jul 11 17:13 tokenizer.json
-rw------- 1 root root 801269 Jul 11 17:13 dpo_pairs.jsonl


In [7]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MAX_SEQ_LENGTH = 1024
SFT_ADAPTER_PATH = "llama3.1-8b-legal-clause-lora-sft"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

# Re-attach LoRA on top for the DPO stage (continues adapting the same adapter weights)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("SFT model + adapter loaded, ready for DPO.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load llama3.1-8b-legal-clause-lora-sft as a legacy tokenizer.
Unsloth 2026.7.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


SFT model + adapter loaded, ready for DPO.


In [8]:
from datasets import load_dataset

dpo_ds = load_dataset("json", data_files="dpo_pairs.jsonl", split="train")
print(f"Loaded {len(dpo_ds)} preference pairs.")
print("Example:", dpo_ds[0])

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 700 preference pairs.
Example: {'prompt': [{'role': 'system', 'content': 'You are a legal clause classifier. Read the contract clause and respond with exactly one category from this list: Governing Laws, Terminations, Confidentiality, Indemnifications, Assignments, Notices, Severability, Counterparts, Amendments, Survival. Respond with only the category name and nothing else.'}, {'role': 'user', 'content': 'Clause:\nAll notices, requests, demands, consents, instructions or other communications required or permitted hereunder shall be in writing and faxed, mailed, or delivered to each party as follows: (i) if to a Voting Party, at such Voting Party’s address or facsimile number set forth in the Company’s records, or at such other address or facsimile number as a Voting Party shall have furnished the Company in writing; or (ii) if to the Company or CEO, at 6720 N. Scottsdale Road, Suite #390, Scottsdale, Arizona 85253, Attention: Chief Executive Officer, or at such other address a

In [9]:
from trl import DPOTrainer, DPOConfig

dpo_args = DPOConfig(
    output_dir="outputs_dpo",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    learning_rate=5e-6,
    beta=0.1,
    warmup_ratio=0.1,
    logging_steps=10,
    optim="adamw_8bit",
    seed=42,
    report_to="wandb",
    run_name="legal-clause-dpo-llama3.1-8b",
    max_length=768,
    max_prompt_length=512,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,   # only keep the last 2 checkpoints, to save disk space
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,     # None is correct with PEFT: TRL recovers reference behavior by
                         # temporarily disabling the adapter, no separate copy needed
    args=dpo_args,
    train_dataset=dpo_ds,
    processing_class=tokenizer,
)

dpo_trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Extracting prompt in train dataset (num_proc=6):   0%|          | 0/700 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/700 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/700 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 700 | Num Epochs = 2 | Total steps = 88
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
10,0.432913,0.141752,-0.547705,0.925000,0.689457,-10.410335,-26.691370,0.203401,0.178487
20,0.271466,0.089144,-1.187934,0.962500,1.277078,-10.955320,-31.960583,0.296996,0.205121
30,0.168774,0.170252,-1.717762,0.981250,1.888014,-10.188945,-37.355930,0.343468,0.236783
40,0.108073,0.306776,-2.109107,0.993750,2.415883,-8.826655,-41.694710,0.470202,0.326391
50,0.087804,0.413279,-2.404194,0.987179,2.817472,-7.568694,-44.479301,0.464968,0.345341
60,0.062302,0.575450,-2.612021,0.987500,3.187470,-6.120676,-46.459629,0.479085,0.360664
70,0.049070,0.687276,-2.905612,0.993750,3.592889,-5.098504,-49.295803,0.503625,0.430391
80,0.050874,0.729782,-2.972941,0.987500,3.702723,-4.403852,-50.437668,0.475822,0.426849


Unsloth: Restored added_tokens_decoder metadata in outputs_dpo/checkpoint-25/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_dpo/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_dpo/checkpoint-75/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_dpo/checkpoint-88/tokenizer_config.json.


TrainOutput(global_step=88, training_loss=0.14277456734668126, metrics={'train_runtime': 3681.3946, 'train_samples_per_second': 0.38, 'train_steps_per_second': 0.024, 'total_flos': 0.0, 'train_loss': 0.14277456734668126, 'epoch': 2.0})

In [10]:
DPO_ADAPTER_DIR = "llama3.1-8b-legal-clause-lora-dpo"
model.save_pretrained(DPO_ADAPTER_DIR)
tokenizer.save_pretrained(DPO_ADAPTER_DIR)

!zip -rq {DPO_ADAPTER_DIR}.zip {DPO_ADAPTER_DIR}

print(f"DPO adapter saved to: {DPO_ADAPTER_DIR}")
print(f"Zipped as {DPO_ADAPTER_DIR}.zip — download this now via the Files sidebar "
      f"so you don't lose it if the session disconnects.")
print("Step 03 complete. Ready for Step 04 (evaluation: base vs SFT vs DPO).")

Unsloth: Restored added_tokens_decoder metadata in llama3.1-8b-legal-clause-lora-dpo/tokenizer_config.json.


DPO adapter saved to: llama3.1-8b-legal-clause-lora-dpo
Zipped as llama3.1-8b-legal-clause-lora-dpo.zip — download this now via the Files sidebar so you don't lose it if the session disconnects.
Step 03 complete. Ready for Step 04 (evaluation: base vs SFT vs DPO).
